# LSEG Data Pull - Implied Duration


## 0. Setup (inkl. Input-Dateien laden)


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import lseg.data as ld

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 120)

from project_paths import BASE_DIR, DATA_DIR, CACHE_DATA_DIR


# Safety guard: never overwrite euro500.parquet from this notebook.
if not hasattr(pd.DataFrame, "_orig_to_parquet_implied"):
    pd.DataFrame._orig_to_parquet_implied = pd.DataFrame.to_parquet

def _guarded_to_parquet(self, path, *args, **kwargs):
    try:
        target = Path(path).expanduser().resolve()
    except Exception:
        target = Path(str(path))
    try:
        forbidden = (DATA_DIR / "euro500.parquet").expanduser().resolve()
    except Exception:
        forbidden = DATA_DIR / "euro500.parquet"
    if str(target) == str(forbidden):
        raise RuntimeError(
            "Write blocked: euro500.parquet is read-only in LSEG_DataPull_Implied.ipynb"
        )
    return pd.DataFrame._orig_to_parquet_implied(self, path, *args, **kwargs)

if pd.DataFrame.to_parquet is not _guarded_to_parquet:
    pd.DataFrame.to_parquet = _guarded_to_parquet

import hashlib
import json
import random
import re
import time
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]


EURO500_PATH = DATA_DIR / "euro500.parquet"
EURO500_RETURNS_PATH = DATA_DIR / "euro500_index_returns.parquet"
DAILY_RETURNS_IN_INDEX_PATH = DATA_DIR / "euro500_daily_returns.parquet"

for p in [EURO500_PATH, EURO500_RETURNS_PATH, DAILY_RETURNS_IN_INDEX_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

euro500 = pd.read_parquet(EURO500_PATH)
euro500_returns = pd.read_parquet(EURO500_RETURNS_PATH)
daily_returns_euro500_in_index = pd.read_parquet(DAILY_RETURNS_IN_INDEX_PATH)

print("Loaded:")
print("- euro500:", euro500.shape)
print("- euro500_returns:", euro500_returns.shape)
print("- daily_returns_euro500_in_index:", daily_returns_euro500_in_index.shape)



Loaded:
- euro500: (56500, 26)
- euro500_returns: (7016, 8)
- daily_returns_euro500_in_index: (3457796, 13)


## 1. Shares Outstanding

Fehlende Werte in `shares_outstanding` werden imputiert mit:

$$
\text{shares\_outstanding} = \frac{\text{mcap\_eur}}{\text{PriceClose}}
$$


In [2]:
# Step 1 - Missing shares_outstanding mit mcap_eur / PriceClose auffuellen
required_cols = ["shares_outstanding", "mcap_eur", "PriceClose"]
missing_required = [c for c in required_cols if c not in euro500.columns]
if missing_required:
    raise KeyError(f"Missing required columns for Step 1: {missing_required}")

for c in required_cols:
    euro500[c] = pd.to_numeric(euro500[c], errors="coerce")

n_total = len(euro500)
missing_before = int(euro500["shares_outstanding"].isna().sum())
coverage_before = 1.0 - (missing_before / n_total) if n_total else np.nan

# Nur dort imputen, wo shares_outstanding fehlt und Divisor valide ist.
fill_mask = (
    euro500["shares_outstanding"].isna()
    & euro500["mcap_eur"].notna()
    & euro500["PriceClose"].notna()
    & (euro500["PriceClose"] != 0)
)

euro500.loc[fill_mask, "shares_outstanding"] = (
    euro500.loc[fill_mask, "mcap_eur"] / euro500.loc[fill_mask, "PriceClose"]
)

missing_after = int(euro500["shares_outstanding"].isna().sum())
coverage_after = 1.0 - (missing_after / n_total) if n_total else np.nan
filled_rows = int(fill_mask.sum())

print("Step 1 coverage update (shares_outstanding):")
print(f"- total rows: {n_total}")
print(f"- missing before: {missing_before} ({(1-coverage_before)*100:.2f}%)")
print(f"- missing after : {missing_after} ({(1-coverage_after)*100:.2f}%)")
print(f"- filled rows   : {filled_rows}")
print(f"- coverage before: {coverage_before*100:.2f}%")
print(f"- coverage after : {coverage_after*100:.2f}%")


# Persist Step 1 base panel for all subsequent steps (single-output design)
if "EURO500_IMPLIED_PATH" not in globals():
    EURO500_IMPLIED_PATH = DATA_DIR / "euro500_implied.parquet"
euro500.to_parquet(EURO500_IMPLIED_PATH, index=False)
print("Saved Step 1 base panel:", EURO500_IMPLIED_PATH, "rows:", len(euro500))



Step 1 coverage update (shares_outstanding):
- total rows: 56500
- missing before: 18025 (31.90%)
- missing after : 368 (0.65%)
- filled rows   : 17657
- coverage before: 68.10%
- coverage after : 99.35%
Saved Step 1 base panel: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_implied.parquet rows: 56500


## 2. Risk free rate


In [3]:
# Step 2 - Market risk-free rate merge (quarter-start as-of, per firm)
# Strict requirement: use annualized €STR only (rf_estr_annual).
RF_SOURCE_PRIMARY = DATA_DIR / "euro500_index_returns.parquet"
RF_CACHE_DIR = CACHE_DATA_DIR / "rf_cache"
IMPLIED_PATH = DATA_DIR / "euro500_implied.parquet"
TARGET_PATH = IMPLIED_PATH

if not TARGET_PATH.exists():
    raise FileNotFoundError(f"Missing target file: {TARGET_PATH}")

tgt = pd.read_parquet(TARGET_PATH).copy()
if "date" not in tgt.columns:
    raise KeyError("'date' column missing in target implied table")

rf_series = None
rf_origin = None

# 1) Preferred source: euro500_index_returns.parquet with rf_estr_annual
if RF_SOURCE_PRIMARY.exists():
    src = pd.read_parquet(RF_SOURCE_PRIMARY).copy()
    if "date" in src.columns and "rf_estr_annual" in src.columns:
        tmp = src[["date", "rf_estr_annual"]].copy()
        tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce").dt.normalize()
        tmp["rf_estr_annual"] = pd.to_numeric(tmp["rf_estr_annual"], errors="coerce")
        tmp = tmp[tmp["date"].notna() & tmp["rf_estr_annual"].notna()].sort_values("date").drop_duplicates("date", keep="last")
        if len(tmp):
            rf_series = tmp
            rf_origin = str(RF_SOURCE_PRIMARY)

# 2) Fallback: newest RF cache file from Euro500_IndexReturns Step 2
if rf_series is None and RF_CACHE_DIR.exists():
    cache_files = sorted(RF_CACHE_DIR.glob("rf_estr_v2_*.parquet"))
    if cache_files:
        cache = pd.read_parquet(cache_files[-1]).copy()
        if "date" in cache.columns and "rf_estr_annual" in cache.columns:
            tmp = cache[["date", "rf_estr_annual"]].copy()
            tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce").dt.normalize()
            tmp["rf_estr_annual"] = pd.to_numeric(tmp["rf_estr_annual"], errors="coerce")
            tmp = tmp[tmp["date"].notna() & tmp["rf_estr_annual"].notna()].sort_values("date").drop_duplicates("date", keep="last")
            if len(tmp):
                rf_series = tmp
                rf_origin = str(cache_files[-1])

if rf_series is None:
    raise KeyError(
        "rf_estr_annual not found. Run Euro500_IndexReturns Step 2 first and ensure "
        "rf_estr_annual is saved in euro500_index_returns.parquet or rf_cache."
    )

# Requested key: for each firm row, use quarter-start and attach current market annual RF rate.
tgt["date"] = pd.to_datetime(tgt["date"], errors="coerce").dt.normalize()
tgt["quarter_start_date"] = tgt["date"].dt.to_period("Q").dt.start_time.dt.normalize()

left = tgt.sort_values("quarter_start_date").copy()
right = rf_series.rename(columns={"date": "rf_obs_date", "rf_estr_annual": "market_risk_free_rate_annual"}).sort_values("rf_obs_date")

left = pd.merge_asof(
    left,
    right,
    left_on="quarter_start_date",
    right_on="rf_obs_date",
    direction="backward",
)

out = left.sort_index().copy()
out["market_risk_free_rate_annual"] = pd.to_numeric(out["market_risk_free_rate_annual"], errors="coerce")

# Explicitly remove any daily RF carry-over columns from implied output.
for c in ["rf_daily", "market_risk_free_rate", "daily_rf", "rf_obs_date", "quarter_start_date"]:
    if c in out.columns:
        out = out.drop(columns=[c])

# Persist only the unified implied table (single-output design).
out.to_parquet(IMPLIED_PATH, index=False)

print("RF origin:", rf_origin)
print("RF column used: rf_estr_annual")
print("Target input:", TARGET_PATH)
print("Rows:", len(out), "| annual rf non-null share:", float(out["market_risk_free_rate_annual"].notna().mean()))




RF origin: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/cache/rf_cache/rf_estr_v2_19980102_20251231.parquet
RF column used: rf_estr_annual
Target input: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_implied.parquet
Rows: 56500 | annual rf non-null share: 0.9469026548672567
